# Daily Challenge: Multi-Attention & Transformer Comparisons
## Week 7 — Day 2 | DI GenAI & Machine Learning Bootcamp 2026

**Objectives:**
- Implement scaled dot-product attention from scratch in PyTorch
- Extend to multi-head attention with residual connections and dropout
- Build a full custom encoder stack and train it on NLI
- Visualize attention maps and compare with a pretrained DistilBERT baseline

**Parts:**
1. Setup & Dataset Loading
2. Single-Head Attention Implementation
3. Multi-Head Attention Module
4. Custom Encoder Stack & Training Loop
5. Attention Map Visualization
6. Pretrained DistilBERT Baseline Comparison
7. Reflection

> **Run on Google Colab** — GPU recommended.

## Part 1 — Setup & Dataset Loading

In [ ]:
!pip install --quiet datasets transformers torch matplotlib seaborn

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset as TorchDataset
from torch.optim import AdamW
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import warnings
warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} ✓")
print(f"Device  : {device}")

In [ ]:
# Load the SNLI (Stanford Natural Language Inference) dataset
# Labels: 0=entailment, 1=neutral, 2=contradiction
print("Loading SNLI dataset...")
raw = load_dataset("snli", split={"train": "train[:8000]", "validation": "validation[:2000]"})

# Drop examples with label == -1 (unlabeled)
raw = raw.filter(lambda x: x["label"] != -1)

LABEL_NAMES = {0: "entailment", 1: "neutral", 2: "contradiction"}
NUM_CLASSES  = 3

print(f"\nDataset loaded ✓")
print(f"  Train      : {len(raw['train']):,} examples")
print(f"  Validation : {len(raw['validation']):,} examples")
print(f"  Labels     : {LABEL_NAMES}")
print(f"\nSample:")
ex = raw['train'][0]
print(f"  Premise    : {ex['premise']}")
print(f"  Hypothesis : {ex['hypothesis']}")
print(f"  Label      : {LABEL_NAMES[ex['label']]} ({ex['label']})")

## Part 2 — Single-Head Attention Implementation

In [ ]:
class SingleHeadAttention(nn.Module):
    """Scaled dot-product attention with learned Q, K, V projections."""

    def __init__(self, embed_dim: int, dropout: float = 0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.scale     = math.sqrt(embed_dim)

        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        # (B, T, D) → projected Q, K, V
        Q = self.W_q(query)  # (B, T_q, D)
        K = self.W_k(key)    # (B, T_k, D)
        V = self.W_v(value)  # (B, T_k, D)

        # Attention scores: (B, T_q, T_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)   # (B, T_q, T_k)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, V)    # (B, T_q, D)
        output  = self.W_o(context)                # (B, T_q, D)

        return output, attn_weights


# ── Shape validation ────────────────────────────────────────────────────────
EMBED_DIM = 64
SEQ_LEN   = 10
BATCH     = 4

dummy = torch.randn(BATCH, SEQ_LEN, EMBED_DIM)
sha   = SingleHeadAttention(embed_dim=EMBED_DIM)
out, weights = sha(dummy, dummy, dummy)

print("Single-Head Attention — shape check")
print(f"  Input          : {tuple(dummy.shape)}")
print(f"  Output         : {tuple(out.shape)}   (should be {BATCH}×{SEQ_LEN}×{EMBED_DIM})")
print(f"  Attention map  : {tuple(weights.shape)}   (should be {BATCH}×{SEQ_LEN}×{SEQ_LEN})")
assert out.shape     == (BATCH, SEQ_LEN, EMBED_DIM), "output shape mismatch"
assert weights.shape == (BATCH, SEQ_LEN, SEQ_LEN),   "attention weight shape mismatch"
print("\nWeight stats (first sample, first row):")
print(f"  sum  = {weights[0, 0].sum().item():.4f}  (should be ≈1.0)")
print(f"  min  = {weights[0, 0].min().item():.4f}")
print(f"  max  = {weights[0, 0].max().item():.4f}")

total_params = sum(p.numel() for p in sha.parameters())
print(f"\nLearnable parameters : {total_params:,}  ({4} matrices × {EMBED_DIM}×{EMBED_DIM})")
print("✓ Single-head attention validated")

## Part 3 — Multi-Head Attention Module

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-head attention: splits embed_dim into num_heads parallel heads,
    runs scaled dot-product attention on each, then concatenates and projects.
    """

    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.embed_dim  = embed_dim
        self.num_heads  = num_heads
        self.head_dim   = embed_dim // num_heads          # d_k per head
        self.scale      = math.sqrt(self.head_dim)

        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.proj_dropout = nn.Dropout(dropout)

    def _split_heads(self, x):
        """(B, T, D) → (B, H, T, d_k)"""
        B, T, D = x.shape
        x = x.reshape(B, T, self.num_heads, self.head_dim)
        return x.transpose(1, 2)   # (B, H, T, d_k)

    def forward(self, query, key, value, mask=None):
        B = query.size(0)

        Q = self._split_heads(self.W_q(query))   # (B, H, T_q, d_k)
        K = self._split_heads(self.W_k(key))     # (B, H, T_k, d_k)
        V = self._split_heads(self.W_v(value))   # (B, H, T_k, d_k)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # (B, H, T_q, T_k)

        if mask is not None:
            # mask: (B, 1, 1, T_k) or (B, 1, T_q, T_k)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)   # (B, H, T_q, T_k)
        attn_weights = self.attn_dropout(attn_weights)

        context = torch.matmul(attn_weights, V)    # (B, H, T_q, d_k)
        # Merge heads: (B, H, T_q, d_k) → (B, T_q, D)
        context = context.transpose(1, 2).reshape(B, -1, self.embed_dim)

        output = self.proj_dropout(self.W_o(context))   # (B, T_q, D)
        return output, attn_weights


# ── Shape validation ────────────────────────────────────────────────────────
NUM_HEADS = 8

dummy = torch.randn(BATCH, SEQ_LEN, EMBED_DIM)
mha   = MultiHeadAttention(embed_dim=EMBED_DIM, num_heads=NUM_HEADS)
out_m, weights_m = mha(dummy, dummy, dummy)

print("Multi-Head Attention — shape check")
print(f"  num_heads      : {NUM_HEADS}")
print(f"  head_dim       : {EMBED_DIM // NUM_HEADS}")
print(f"  Input          : {tuple(dummy.shape)}")
print(f"  Output         : {tuple(out_m.shape)}   (should be {BATCH}×{SEQ_LEN}×{EMBED_DIM})")
print(f"  Attention maps : {tuple(weights_m.shape)}   (should be {BATCH}×{NUM_HEADS}×{SEQ_LEN}×{SEQ_LEN})")
assert out_m.shape     == (BATCH, SEQ_LEN, EMBED_DIM),           "output shape mismatch"
assert weights_m.shape == (BATCH, NUM_HEADS, SEQ_LEN, SEQ_LEN), "attn weight shape mismatch"

total_m = sum(p.numel() for p in mha.parameters())
print(f"\nLearnable parameters : {total_m:,}")
print(f"  (4 matrices × {EMBED_DIM}×{EMBED_DIM} = {4 * EMBED_DIM * EMBED_DIM:,})")
print("✓ Multi-head attention validated")

# ── Residual + LayerNorm test ─────────────────────────────────────────────
norm = nn.LayerNorm(EMBED_DIM)
residual_out = norm(dummy + out_m)     # Add & Norm
print(f"\nResidual + LayerNorm output shape : {tuple(residual_out.shape)}  ✓")

## Part 4 — Custom Encoder Stack & Training Loop

In [ ]:
# ── Tokenizer + NLI Dataset ──────────────────────────────────────────────────
from transformers import AutoTokenizer as HFTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN    = 64

hf_tok = HFTokenizer.from_pretrained(MODEL_NAME)
VOCAB_SIZE = hf_tok.vocab_size

def encode_nli(batch):
    """Tokenize premise+hypothesis pair as a single sequence."""
    return hf_tok(
        batch["premise"],
        batch["hypothesis"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors=None,
    )

tok_train = raw["train"].map(encode_nli, batched=True, remove_columns=["premise","hypothesis"])
tok_val   = raw["validation"].map(encode_nli, batched=True, remove_columns=["premise","hypothesis"])
tok_train.set_format(type="torch", columns=["input_ids","attention_mask","label"])
tok_val.set_format(type="torch",   columns=["input_ids","attention_mask","label"])

print(f"Tokenizer : {MODEL_NAME}  (vocab={VOCAB_SIZE:,})")
print(f"Train      : {len(tok_train):,} examples")
print(f"Validation : {len(tok_val):,} examples")
sample = tok_train[0]
print(f"input_ids  shape : {sample['input_ids'].shape}")
print(f"labels     : {sample['label'].item()} ({LABEL_NAMES[sample['label'].item()]})")

In [ ]:
class EncoderLayer(nn.Module):
    """One Transformer encoder layer: MHA → Add&Norm → FFN → Add&Norm."""

    def __init__(self, embed_dim: int, num_heads: int, ffn_dim: int, dropout: float = 0.1):
        super().__init__()
        self.mha   = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, mask=None):
        attn_out, attn_weights = self.mha(x, x, x, mask)
        x = self.norm1(x + attn_out)          # residual 1
        x = self.norm2(x + self.ffn(x))       # residual 2
        return x, attn_weights


class NLIEncoder(nn.Module):
    """
    Custom Transformer encoder for 3-class NLI.
    Embedding → N encoder layers → mean pool → classifier head.
    """

    def __init__(self, vocab_size, embed_dim, num_heads, ffn_dim,
                 num_layers, num_classes, max_len=128, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # Learnable positional embeddings
        self.pos_emb   = nn.Embedding(max_len, embed_dim)
        self.drop_emb  = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            EncoderLayer(embed_dim, num_heads, ffn_dim, dropout)
            for _ in range(num_layers)
        ])
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, num_classes),
        )

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        pos   = torch.arange(T, device=input_ids.device).unsqueeze(0)  # (1, T)
        x     = self.drop_emb(self.embedding(input_ids) + self.pos_emb(pos))

        # Expand mask for attention: (B, 1, 1, T)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(1).unsqueeze(2)
        else:
            mask = None

        all_attn = []
        for layer in self.layers:
            x, attn = layer(x, mask)
            all_attn.append(attn)

        # Mean-pool over non-padding tokens
        if attention_mask is not None:
            lengths = attention_mask.sum(dim=1, keepdim=True).unsqueeze(-1).float()
            pooled  = (x * attention_mask.unsqueeze(-1).float()).sum(1) / lengths.squeeze(-1)
        else:
            pooled = x.mean(dim=1)

        logits = self.classifier(pooled)
        return logits, all_attn


# ── Instantiate ───────────────────────────────────────────────────────────────
EMBED_DIM_M  = 128
NUM_HEADS_M  = 4
FFN_DIM_M    = 256
NUM_LAYERS   = 2
DROPOUT      = 0.1

custom_model = NLIEncoder(
    vocab_size  = VOCAB_SIZE,
    embed_dim   = EMBED_DIM_M,
    num_heads   = NUM_HEADS_M,
    ffn_dim     = FFN_DIM_M,
    num_layers  = NUM_LAYERS,
    num_classes = NUM_CLASSES,
    max_len     = MAX_LEN,
    dropout     = DROPOUT,
).to(device)

n_params = sum(p.numel() for p in custom_model.parameters() if p.requires_grad)
print(f"NLIEncoder : {n_params:,} trainable parameters")
print(f"  Layers   : {NUM_LAYERS} × EncoderLayer")
print(f"  Heads    : {NUM_HEADS_M}  |  head_dim = {EMBED_DIM_M // NUM_HEADS_M}")
print(f"  FFN dim  : {FFN_DIM_M}")

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64
LR         = 3e-4
EPOCHS     = 5

train_loader = DataLoader(tok_train, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(tok_val,   batch_size=BATCH_SIZE)

optimizer  = AdamW(custom_model.parameters(), lr=LR, weight_decay=1e-2)
criterion  = nn.CrossEntropyLoss()

train_losses, val_losses   = [], []
train_accs,   val_accs     = [], []

for epoch in range(1, EPOCHS + 1):
    # ── Training ───────────────────────────────────────────────────────────
    custom_model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch in train_loader:
        ids    = batch["input_ids"].to(device)
        mask   = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits, _ = custom_model(ids, mask)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(custom_model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * ids.size(0)
        preds       = logits.argmax(dim=-1)
        correct    += (preds == labels).sum().item()
        total      += ids.size(0)

    train_loss = total_loss / total
    train_acc  = correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # ── Validation ────────────────────────────────────────────────────────
    custom_model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0

    with torch.no_grad():
        for batch in val_loader:
            ids    = batch["input_ids"].to(device)
            mask   = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits, _ = custom_model(ids, mask)
            loss = criterion(logits, labels)
            v_loss   += loss.item() * ids.size(0)
            preds     = logits.argmax(dim=-1)
            v_correct += (preds == labels).sum().item()
            v_total   += ids.size(0)

    val_loss = v_loss / v_total
    val_acc  = v_correct / v_total
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"Epoch {epoch}/{EPOCHS}  "
          f"train loss={train_loss:.4f}  acc={train_acc:.3f}  |  "
          f"val loss={val_loss:.4f}  acc={val_acc:.3f}")

# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, train_losses, 'b-o', label='Train')
axes[0].plot(epochs_range, val_losses,   'r-o', label='Val')
axes[0].set_title('Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, train_accs, 'b-o', label='Train')
axes[1].plot(epochs_range, val_accs,   'r-o', label='Val')
axes[1].set_title('Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Custom NLI Encoder — Training', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('custom_encoder_training.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"\nFinal val accuracy : {val_accs[-1]:.3f}")
print("Plot saved ✓")

## Part 5 — Attention Map Visualization

In [ ]:
def get_attention_maps(model, input_ids, attention_mask):
    """Run one forward pass and return attention weights from all layers."""
    model.eval()
    with torch.no_grad():
        _, all_attn = model(
            input_ids.unsqueeze(0).to(device),
            attention_mask.unsqueeze(0).to(device),
        )
    # each element: (1, H, T, T) → (H, T, T)
    return [a.squeeze(0).cpu() for a in all_attn]


# Pick a sample from validation set
sample_idx = 5
sample_ids  = tok_val[sample_idx]["input_ids"]
sample_mask = tok_val[sample_idx]["attention_mask"]
sample_label = tok_val[sample_idx]["label"].item()

# Decode tokens (drop padding)
seq_len = int(sample_mask.sum().item())
tokens_display = hf_tok.convert_ids_to_tokens(sample_ids[:seq_len].tolist())
tokens_display = [t.replace("##", "") for t in tokens_display]   # clean sub-word markers

all_maps = get_attention_maps(custom_model, sample_ids, sample_mask)

# ── Plot: all heads for layer 0 ───────────────────────────────────────────────
LAYER_TO_PLOT = 0
attn_layer    = all_maps[LAYER_TO_PLOT]  # (H, T, T)

fig, axes = plt.subplots(1, NUM_HEADS_M, figsize=(4 * NUM_HEADS_M, 4))
if NUM_HEADS_M == 1:
    axes = [axes]

for h, ax in enumerate(axes):
    map_h = attn_layer[h, :seq_len, :seq_len].numpy()   # (T_vis, T_vis)
    sns.heatmap(map_h, ax=ax, cmap="Blues", vmin=0, vmax=map_h.max(),
                xticklabels=tokens_display, yticklabels=tokens_display,
                cbar=(h == NUM_HEADS_M - 1))
    ax.set_title(f"Head {h+1}", fontsize=9, fontweight='bold')
    ax.tick_params(axis='both', labelsize=6)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.suptitle(f"Layer {LAYER_TO_PLOT+1} Attention Maps  |  Label: {LABEL_NAMES[sample_label]}",
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('attn_map_layer1_all_heads.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

# ── Plot: mean attention across all heads, both layers ─────────────────────
fig, axes = plt.subplots(1, len(all_maps), figsize=(6 * len(all_maps), 5))
if len(all_maps) == 1:
    axes = [axes]

for l_idx, (ax, attn_l) in enumerate(zip(axes, all_maps)):
    mean_map = attn_l[:, :seq_len, :seq_len].mean(dim=0).numpy()
    sns.heatmap(mean_map, ax=ax, cmap="YlOrRd", vmin=0, vmax=mean_map.max(),
                xticklabels=tokens_display, yticklabels=tokens_display)
    ax.set_title(f"Layer {l_idx+1} — Mean over {NUM_HEADS_M} heads", fontweight='bold')
    ax.tick_params(axis='both', labelsize=7)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.suptitle(f"Mean Attention  |  Label: {LABEL_NAMES[sample_label]}", fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('attn_map_mean_all_layers.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

## Part 6 — Pretrained DistilBERT Baseline Comparison

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

# ── Load DistilBERT fine-tuned on NLI ─────────────────────────────────────────
distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_CLASSES,
).to(device)

db_params = sum(p.numel() for p in distilbert_model.parameters())
print(f"DistilBERT parameters : {db_params:,}")
print(f"Custom encoder params : {n_params:,}")
print(f"Size ratio            : {db_params / n_params:.1f}×")

# ── Fine-tune DistilBERT for 3 epochs ─────────────────────────────────────────
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return acc_metric.compute(predictions=preds, references=labels)

# Rename 'label' → 'labels' (Trainer expects 'labels')
db_train = tok_train.rename_column("label", "labels")
db_val   = tok_val.rename_column("label",   "labels")

training_args = TrainingArguments(
    output_dir               = "./distilbert_nli",
    num_train_epochs         = 3,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size  = 64,
    learning_rate            = 2e-5,
    weight_decay             = 0.01,
    eval_strategy            = "epoch",
    save_strategy            = "no",
    logging_steps            = 50,
    report_to                = "none",
    fp16                     = torch.cuda.is_available(),
)

trainer = Trainer(
    model           = distilbert_model,
    args            = training_args,
    train_dataset   = db_train,
    eval_dataset    = db_val,
    compute_metrics = compute_metrics,
)

print("\nFine-tuning DistilBERT on SNLI (3 epochs)…")
train_result = trainer.train()
eval_result  = trainer.evaluate()

db_val_acc = eval_result["eval_accuracy"]
print(f"\nDistilBERT val accuracy : {db_val_acc:.3f}")

In [ ]:
# ── Side-by-side comparison bar chart ────────────────────────────────────────
custom_val_acc  = val_accs[-1]
distilbert_acc  = db_val_acc
random_baseline = 1 / NUM_CLASSES   # 0.333…

models  = ["Random\nBaseline", "Custom Encoder\n(scratch)", "DistilBERT\n(fine-tuned)"]
accs    = [random_baseline,    custom_val_acc,              distilbert_acc]
colors  = ["#bdbdbd",          "#4e91d4",                   "#e07b39"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(models, accs, color=colors, edgecolor='white', width=0.5)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Validation Accuracy")
ax.set_title("NLI Accuracy Comparison", fontweight='bold', fontsize=13)
ax.axhline(random_baseline, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax.grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.015,
            f"{acc:.3f}", ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Comparison plot saved ✓")

print("\n─── Summary ────────────────────────────────────────────")
print(f"  Random baseline        : {random_baseline:.3f}")
print(f"  Custom encoder (5 ep)  : {custom_val_acc:.3f}")
print(f"  DistilBERT (3 ep FT)   : {distilbert_acc:.3f}")
print(f"  Gap (custom vs DB)     : {distilbert_acc - custom_val_acc:+.3f}")
print("────────────────────────────────────────────────────────")

## Part 7 — Reflection

### Single-Head vs Multi-Head vs Cross-Attention

| Aspect | Single-Head | Multi-Head | Cross-Attention |
|---|---|---|---|
| **What it does** | One set of Q/K/V projections; attends globally | H parallel heads each with d_k = d/H; projections merged | Q from one sequence (decoder), K/V from another (encoder output) |
| **Pattern diversity** | One attention pattern per layer | H independent patterns — each head can focus on different relationships (syntax, coreference, semantics) | Bridges two sequences; used in translation, summarization, captioning |
| **Parameter cost** | 4 × d² | Same — 4 × d² total (just split into heads) | Same as MHA but queries come from a different source |
| **Expressiveness** | Limited — single linear subspace | Higher — multiple subspaces in parallel | Needed whenever output must condition on a separate input sequence |
| **Typical use** | Simple baselines, ablations | Every Transformer encoder layer (BERT, RoBERTa, etc.) | Encoder-decoder Transformers (T5, BART, MarianMT) |

### Key Observations from This Challenge

1. **Shape algebra**: the multi-head split `(B, T, D) → (B, H, T, d/H)` is just a reshape + transpose — no extra parameters, but it unlocks H parallel representation subspaces.

2. **Residual connections are essential**: without `x + attn_out` and `x + ffn(x)`, deep stacks suffer from vanishing gradients. LayerNorm stabilizes the scale after each residual addition.

3. **Custom encoder vs DistilBERT**: the accuracy gap illustrates the power of pretraining — DistilBERT's weights encode billions of tokens of linguistic knowledge before task fine-tuning even begins. Our model trains from random initialization on only 8,000 examples.

4. **Attention maps reveal specialization**: different heads learn to attend to different token pairs — some focus on adjacent tokens (local syntax), others connect distant subject-verb pairs or [CLS]/[SEP] boundaries. Mean-pooling across heads gives a smoother picture of overall token salience.

5. **Padding masking matters**: setting `−∞` on padding positions before softmax ensures those positions contribute zero to the weighted sum, preventing "attention leakage" onto meaningless pad tokens.